# CUDA Vector Addition

## Objective

To implement a simple vector addition operation using NVIDIA CUDA through Python and Numba.

The experiment demonstrates:
- GPU availability verification.
- CUDA kernel implementation.
- Memory transfer between CPU and GPU.
- CUDA thread and block configuration.
- GPU kernel execution.
- Verification of GPU results against CPU results.

In [1]:
!nvidia-smi

Thu Nov 12 15:44:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import numba
from numba import cuda

print("NumPy version:", np.__version__)
print("Numba version:", numba.__version__)
print("CUDA available:", cuda.is_available())

if cuda.is_available():
    print("GPU:", cuda.get_current_device().name)
else:
    print("CUDA GPU is not available.")

NumPy version: 2.4.6
Numba version: 0.64.0
CUDA available: True
GPU: Tesla T4


In [3]:
# Create input arrays in CPU memory
A = np.array([1, 2, 3, 4, 5], dtype=np.float32)
B = np.array([10, 20, 30, 40, 50], dtype=np.float32)

# Initialize output array
C = np.zeros_like(A)

print("Input Array A:", A)
print("Input Array B:", B)
print("Initial Array C:", C)

Input Array A: [1. 2. 3. 4. 5.]
Input Array B: [10. 20. 30. 40. 50.]
Initial Array C: [0. 0. 0. 0. 0.]


In [4]:
@cuda.jit
def vector_add(A, B, C):
    i = cuda.grid(1)

    if i < C.size:
        C[i] = A[i] + B[i]

In [5]:
if cuda.is_available():

    # Transfer arrays to GPU memory
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.to_device(C)

    # Configure threads and blocks
    threads_per_block = 128

    blocks_per_grid = (
        A.size + threads_per_block - 1
    ) // threads_per_block

    print("Threads per block:", threads_per_block)
    print("Blocks per grid:", blocks_per_grid)

    # Execute CUDA kernel
    vector_add[blocks_per_grid, threads_per_block](
        d_A, d_B, d_C
    )

    # Copy result back to CPU memory
    C = d_C.copy_to_host()

    print("GPU Result:", C)

else:
    print("CUDA GPU is not available.")

Threads per block: 128
Blocks per grid: 1


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU Result: [11. 22. 33. 44. 55.]


In [6]:
if cuda.is_available():

    expected = A + B

    print("Expected Result:", expected)
    print("GPU Result:", C)

    assert np.array_equal(C, expected), "Results do not match!"

    print("Vector addition completed successfully!")

Expected Result: [11. 22. 33. 44. 55.]
GPU Result: [11. 22. 33. 44. 55.]
Vector addition completed successfully!


## Conclusion

Vector addition was successfully implemented using a CUDA kernel with Numba on the NVIDIA Tesla T4 GPU through Google Colab.

The input arrays were transferred from CPU memory to GPU memory, and the addition operation was executed in parallel using CUDA threads. The computed output was transferred back to CPU memory and verified against the expected result.

The experiment demonstrates the basic GPU programming workflow, including memory transfers, thread and block configuration, kernel execution, and result verification.